In [0]:
%pip install --upgrade "mlflow[databricks]>=3.1" plotly
dbutils.library.restartPython()


# Chapter 6: Evaluating GenAI Applications within MLflow 3+ on Databricks

**Learning Objective:** Understand and apply evaluation techniques for GenAI applications using MLflow.

## Table of Contents
1. The Modern Evaluation Landscape in MLflow
   - Why traditional metrics miss GenAI application behaviour
   - What are the components to evaluate
   - Evaluation modes: Direct evaluation vs Answer sheet evaluation
2. Creating & Managing Evaluation Datasets
   - Build or import datasets from scratch for targeted tests
   - Synthesize evaluation sets
3. Scores
   - How scores work
   - Use LLM-based scores
   - Use code-based scores
   - How to evaluate different components
4. Human Feedback
   - Understanding the Feedback data model
   - End user feedback collection
   - Analyzing feedback data
5. Evaluation Runs
   - Understanding evaluation runs
   - Creating evaluation runs
   - View and interpret results
6. Best Practices

## Use Case: Unity Airways Customer Service Agent

This notebook demonstrates best practices for evaluating a GenAI-powered customer service agent for Unity Airways (a fictional airline). We'll cover all aspects of evaluation from traditional metrics limitations to modern MLflow 3+ evaluation techniques.


---

## 1. The Modern Evaluation Landscape in MLflow {#modern-evaluation}

GenAI applications represent a fundamental shift from traditional ML models. Unlike classic models that produce structured, deterministic outputs, GenAI systems generate natural language responses that must be evaluated for qualities like relevance, helpfulness, factual accuracy, and user satisfaction.

### Why Traditional Metrics Miss GenAI Application Behaviour

Traditional ML evaluation metrics (accuracy, precision, recall, F1-score) were designed for classification and regression tasks with clear ground truth labels. These metrics fall short for GenAI applications because:

1. **Deterministic vs. Generative Outputs**: Traditional models produce fixed outputs for given inputs, while GenAI models generate variable, creative responses
2. **Structured vs. Unstructured Data**: Traditional metrics work with numerical or categorical outputs, not natural language text
3. **Single Correct Answer vs. Multiple Valid Responses**: GenAI tasks often have many acceptable answers, making binary accuracy insufficient
4. **Context and Nuance**: Traditional metrics don't capture semantic meaning, tone, helpfulness, or user experience
5. **Safety and Ethics**: GenAI outputs must be evaluated for harmful content, bias, and policy compliance
6. **User Experience**: Traditional metrics don't measure user satisfaction, task completion, or perceived quality

### What are the Components to Evaluate

GenAI applications typically consist of multiple components that require different evaluation approaches:

#### 1. Retrieval Components
- **Retrieval accuracy and relevance**: Do retrieved documents match the query intent?
- **Document ranking quality**: Are the most relevant documents ranked highest?
- **Coverage of relevant information**: Does the retrieved content cover all aspects needed?

#### 2. Generation Components
- **Factual correctness and groundedness**: Are responses factually accurate and based on provided context?
- **Relevance to user query**: Does the response directly address what was asked?
- **Coherence and fluency**: Is the response well-structured and readable?
- **Tone and style appropriateness**: Does the response match the expected communication style?

#### 3. End-to-End System
- **User experience and satisfaction**: Are users happy with the overall interaction?
- **Task completion effectiveness**: Can users accomplish their goals?
- **Safety and policy compliance**: Are responses safe and compliant with guidelines?
- **Latency and performance**: Does the system respond quickly enough?

#### 4. Business Logic
- **Adherence to guidelines and policies**: Does the system follow company rules?
- **Consistency across similar queries**: Are similar questions answered consistently?
- **Integration with downstream systems**: Does the system work well with other tools?

### Evaluation Modes: Direct Evaluation vs Answer Sheet Evaluation

#### Direct Evaluation
- **What it is**: Assesses model outputs directly against criteria or guidelines without comparing to specific reference answers
- **When to use**: Open-ended tasks without single correct answers
- **How it works**: Uses LLM judges to evaluate qualities like helpfulness, relevance, safety
- **Examples**: Chatbot responses, creative writing, summarization, customer service interactions
- **Advantages**: Flexible, can evaluate subjective qualities, works with creative tasks
- **Disadvantages**: Can be inconsistent, requires careful prompt engineering

#### Answer Sheet Evaluation
- **What it is**: Compares model outputs to curated reference answers or ground truth
- **When to use**: Tasks with clear correct answers or specific expected responses
- **How it works**: Uses exact match, semantic similarity, or custom comparison functions
- **Examples**: Question answering, fact extraction, classification, policy lookup
- **Advantages**: Objective, reproducible, easy to understand
- **Disadvantages**: Limited to tasks with clear right/wrong answers, may miss acceptable variations

#### Choosing the Right Mode
- Use **Direct Evaluation** for conversational AI, creative tasks, and subjective quality assessment
- Use **Answer Sheet Evaluation** for factual questions, policy compliance, and objective correctness
- Often combine both approaches for comprehensive evaluation


---

## 2. Creating & Managing Evaluation Datasets {#evaluation-datasets}

Evaluation datasets are the foundation of GenAI application testing. MLflow 3+ provides powerful tools for creating, managing, and versioning evaluation datasets that enable systematic testing and continuous improvement.

### Build or Import Datasets from Scratch for Targeted Tests

#### Creating MLflow Evaluation Datasets

```python
import mlflow.genai.datasets

# Create a new evaluation dataset
eval_dataset = mlflow.genai.datasets.create_dataset(
    uc_table_name="catalog.schema.unity_airways_evaluation",
    name="Unity Airways Customer Service Evaluation",
    description="Comprehensive evaluation dataset for Unity Airways customer service agent"
)
```

#### Building Datasets from Scratch

Create targeted test cases for specific scenarios:

```python
# Define comprehensive test cases for different scenarios
evaluation_examples = [
    # Baggage policy questions
    {
        "inputs": {"question": "What is the baggage allowance for international flights?"},
        "expected": {
            "expected_response": "For international flights, you can bring one carry-on bag (22x14x9 inches) and one personal item. Checked baggage allowance varies by fare type.",
            "category": "baggage",
            "complexity": "simple"
        }
    },
    # Cancellation scenarios
    {
        "inputs": {"question": "How do I cancel my flight and get a refund?"},
        "expected": {
            "expected_response": "You can cancel your flight online through Manage My Booking, by calling customer service, or at the airport. Refund eligibility depends on your fare type and timing.",
            "category": "cancellation",
            "complexity": "medium"
        }
    },
    # Complex multi-part questions
    {
        "inputs": {"question": "I'm traveling with my elderly mother who uses a wheelchair. What assistance is available and what documents do we need?"},
        "expected": {
            "expected_response": "We provide wheelchair assistance and priority boarding. Please request special assistance when booking or at least 48 hours before departure. No additional documentation is required for wheelchair assistance.",
            "category": "accessibility",
            "complexity": "complex"
        }
    }
]

# Add examples to dataset
eval_dataset.merge_records(evaluation_examples)
```

#### Importing Existing Datasets

Import from various formats:

```python
import pandas as pd

# From CSV
csv_data = pd.read_csv("customer_service_qa.csv")
csv_examples = []
for _, row in csv_data.iterrows():
    csv_examples.append({
        "inputs": {"question": row['question']},
        "expected": {"expected_response": row['expected_answer']}
    })

# From JSON
import json
with open("faq_data.json", "r") as f:
    json_data = json.load(f)
    
json_examples = []
for item in json_data:
    json_examples.append({
        "inputs": {"question": item['question']},
        "expected": {"expected_response": item['answer']}
    })

# Merge all sources
eval_dataset.merge_records(csv_examples + json_examples)
```

#### Building from Production Traces

Leverage real user interactions:

```python
import mlflow
import time

# Search for recent successful traces
one_hour_ago = int((time.time() - 60 * 60) * 1000)

traces = mlflow.search_traces(
    filter_string=f"attributes.timestamp_ms > {one_hour_ago} AND "
                 f"attributes.status = 'OK'",
    order_by=["attributes.timestamp_ms DESC"],
    max_results=100
)

# Filter and curate high-quality examples
curated_traces = []
for trace in traces:
    # Add criteria for trace selection
    if trace.info.execution_time_ms < 5000:  # Fast responses
        curated_traces.append(trace)

# Add to evaluation dataset
eval_dataset.merge_records(curated_traces)
```

### Synthesize Evaluation Sets

Generate synthetic data to expand test coverage:

#### Using MLflow's Synthesis Features

```python
from mlflow.genai.datasets import synthesize_dataset

# Generate synthetic customer service scenarios
synthetic_dataset = synthesize_dataset(
    base_examples=evaluation_examples,
    num_examples=100,
    persona_variations=[
        "frustrated customer", 
        "first-time flyer", 
        "business traveler",
        "family with children",
        "international traveler"
    ],
    scenario_types=[
        "booking inquiry",
        "cancellation request", 
        "policy question",
        "complaint handling",
        "special assistance"
    ]
)

eval_dataset.merge_records(synthetic_dataset)
```

#### Custom Synthetic Data Generation

```python
# Custom synthesis for edge cases
edge_case_templates = [
    "What happens if my flight is cancelled due to {reason}?",
    "Can I change my {ticket_type} ticket to {destination}?",
    "I have a {time_constraint} and need to {action}. What are my options?"
]

reasons = ["weather", "mechanical issues", "crew shortage", "air traffic control"]
ticket_types = ["economy", "business", "first class"]
destinations = ["domestic", "international", "connecting"]
time_constraints = ["medical emergency", "business meeting", "family event"]
actions = ["cancel", "reschedule", "upgrade", "get refund"]

synthetic_edge_cases = []
for template in edge_case_templates:
    for i in range(10):  # Generate 10 variations per template
        if "{reason}" in template:
            question = template.format(reason=random.choice(reasons))
        elif "{ticket_type}" in template and "{destination}" in template:
            question = template.format(
                ticket_type=random.choice(ticket_types),
                destination=random.choice(destinations)
            )
        elif "{time_constraint}" in template and "{action}" in template:
            question = template.format(
                time_constraint=random.choice(time_constraints),
                action=random.choice(actions)
            )
        
        synthetic_edge_cases.append({
            "inputs": {"question": question},
            "expected": {"category": "edge_case", "requires_human_review": True}
        })

eval_dataset.merge_records(synthetic_edge_cases)
```

#### Domain Expert Validation

```python
import mlflow.genai.labeling as labeling

# Create labeling session for expert review
expert_session = labeling.create_labeling_session(
    name="Unity Airways Dataset Validation",
    description="Expert validation of synthetic and curated examples",
    assigned_users=["domain_expert@company.com", "qa_lead@company.com"]
)

# Add examples for expert review
expert_session.add_records(synthetic_edge_cases)

# After expert review, sync validated examples
expert_session.sync(dataset_name="catalog.schema.unity_airways_validated")
```

For more details, see: [Build Evaluation Dataset Documentation](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/build-eval-dataset)


---

## 3. Scores {#scores}

Scoring is the heart of GenAI evaluation. MLflow 3+ provides a comprehensive framework for assessing GenAI applications using both automated and human-informed evaluation methods.

### How Scores Work

Scores in MLflow GenAI evaluation are functions that take model inputs and outputs and return a quantitative or qualitative assessment. They operate by:

1. **Input Processing**: Receiving the model's input, output, and optional expected results
2. **Evaluation Logic**: Applying specific criteria or algorithms to assess quality
3. **Score Generation**: Returning numerical scores (0-1, 1-5, etc.), boolean values (pass/fail), or categorical assessments
4. **Rationale Provision**: Providing explanations for the scores to aid in debugging and improvement

**Score Structure:**
```python
{
    "score": 0.85,  # Numerical score
    "justification": "Response is relevant and helpful but lacks specific contact information"
}
```

### Use LLM-Based Scores

LLM-based scorers use large language models as judges to evaluate outputs for complex qualities that require understanding of context, semantics, and nuance.

#### Predefined LLM Scorers

MLflow 3+ provides built-in LLM scorers for common evaluation needs:

```python
from mlflow.genai.scorers import (
    RetrievalGroundedness,    # Checks if response is grounded in retrieved context
    RelevanceToQuery,         # Evaluates relevance to user query  
    Safety,                   # Detects harmful or inappropriate content
    AnswerCorrectness,        # Compares against ground truth answers
    Faithfulness,             # Checks factual consistency with source material
    AnswerRelevance,          # Evaluates answer relevance to the question
    Toxicity,                 # Detects toxic or offensive content
    Coherence                 # Assesses logical flow and readability
)

# Use predefined scorers
predefined_scorers = [
    RetrievalGroundedness(),
    RelevanceToQuery(), 
    Safety(),
    AnswerCorrectness(),
    Faithfulness()
]
```

**Key Predefined Scorers:**
- **RetrievalGroundedness**: Ensures responses are based on provided context
- **RelevanceToQuery**: Measures how well the response addresses the question
- **Safety**: Detects harmful, biased, or inappropriate content
- **AnswerCorrectness**: Compares responses to ground truth answers
- **Faithfulness**: Checks consistency with source material

For complete list and details, see: [Predefined Judge Scorers](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/predefined-judge-scorers)

#### Custom LLM Scorers

Create custom LLM-based scorers for domain-specific evaluation needs:

##### Guidelines-Based Judges

Evaluate outputs against specific guidelines or policies:

```python
from mlflow.genai.scorers import Guidelines

# Professional tone scorer
professional_tone_scorer = Guidelines(
    name="professional_tone",
    guidelines="""
    The customer service response must use professional, courteous language appropriate for airline customer service.
    Requirements:
    - Use polite and respectful language
    - Avoid casual expressions or slang
    - Maintain helpful and solution-oriented tone
    - Include appropriate greetings/closings when relevant
    """
)

# Brand compliance scorer
brand_compliance_scorer = Guidelines(
    name="brand_compliance",
    guidelines="""
    The response must follow Unity Airways brand guidelines:
    - Mention Unity Airways when appropriate
    - Use consistent contact information (1-800-UNITY-AIR, unityairways.com)
    - Maintain positive, customer-focused messaging
    - Provide specific, actionable information when possible
    """
)

# Completeness scorer
completeness_scorer = Guidelines(
    name="response_completeness", 
    guidelines="""
    The customer service response must completely address the customer's question:
    - Directly answer the specific question asked
    - Provide all relevant details mentioned in the expected response
    - Include next steps or additional resources when appropriate
    - Avoid generic responses when specific information is requested
    """
)
```

##### Prompt-Based Judges

Create custom evaluation logic using detailed prompt engineering:

```python
from mlflow.genai.scorers import PromptBasedJudge

# Custom prompt-based scorer for empathy assessment
empathy_scorer = PromptBasedJudge(
    name="empathy_assessment",
    prompt="""
    Evaluate the empathy and emotional intelligence shown in this customer service response.
    
    Customer Question: {question}
    Agent Response: {response}
    
    Consider these aspects:
    1. Does the response acknowledge the customer's feelings or situation?
    2. Is the tone appropriate for the customer's emotional state?
    3. Does the agent show understanding and compassion?
    4. Are any frustrations or concerns addressed sensitively?
    
    Rate on a scale of 1-5 where:
    1 = No empathy shown, cold or dismissive
    2 = Minimal empathy, mostly transactional
    3 = Some empathy shown, adequate emotional response
    4 = Good empathy, shows understanding and care
    5 = Excellent empathy, highly emotionally intelligent response
    
    Provide your rating and explain your reasoning.
    """
)

# Accuracy assessment for policy information
policy_accuracy_scorer = PromptBasedJudge(
    name="policy_accuracy",
    prompt="""
    Evaluate the accuracy of policy information in this airline customer service response.
    
    Customer Question: {question}
    Agent Response: {response}
    Expected Information: {expected}
    
    Check for:
    1. Factual accuracy of policies mentioned
    2. Completeness of policy information
    3. Any contradictions or inconsistencies
    4. Currency of information (not outdated)
    
    Rate as:
    - "accurate" if all policy information is correct and complete
    - "partially_accurate" if mostly correct but missing some details
    - "inaccurate" if contains wrong or misleading information
    
    Explain your assessment.
    """
)
```

For more details, see: 
- [Custom Judge Guidelines](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-judge/meets-guidelines)
- [Create Prompt Judge](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-judge/create-prompt-judge)

### Use Code-Based Scores

Code-based scorers provide deterministic evaluation using Python functions. They're fast, consistent, and ideal for rule-based assessments.

```python
from mlflow.genai.scorers import CodeBasedScorer

def response_length_scorer(prediction, **kwargs):
    """Score based on response length appropriateness."""
    response = prediction if isinstance(prediction, str) else prediction.get('response', '')
    word_count = len(response.split())
    
    if word_count < 10:
        return {"score": 0, "justification": f"Response too short ({word_count} words)"}
    elif word_count > 200:
        return {"score": 0.5, "justification": f"Response quite long ({word_count} words)"}
    else:
        return {"score": 1, "justification": f"Appropriate length ({word_count} words)"}

def contact_info_scorer(prediction, **kwargs):
    """Check if response includes appropriate Unity Airways contact information."""
    response = prediction if isinstance(prediction, str) else prediction.get('response', '')
    response_lower = response.lower()
    
    has_phone = "1-800-unity-air" in response_lower
    has_website = "unityairways.com" in response_lower
    mentions_contact = any(word in response_lower for word in ["call", "phone", "website", "visit"])
    
    if has_phone and has_website:
        return {"score": 1, "justification": "Includes both phone and website"}
    elif has_phone or has_website:
        return {"score": 0.8, "justification": "Includes one form of contact info"}
    elif mentions_contact:
        return {"score": 0.3, "justification": "Mentions contacting but no specific info"}
    else:
        return {"score": 0, "justification": "No contact information provided"}

def policy_keyword_scorer(prediction, expected=None, **kwargs):
    """Check if response contains key policy terms."""
    if not expected:
        return {"score": 0.5, "justification": "No expected response to compare against"}
    
    response = prediction if isinstance(prediction, str) else prediction.get('response', '')
    expected_response = expected.get('expected_response', '') if isinstance(expected, dict) else str(expected)
    
    # Extract key terms (excluding common words)
    response_words = set(response.lower().split())
    expected_words = set(expected_response.lower().split())
    
    # Remove common words
    common_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'are'}
    response_words -= common_words
    expected_words -= common_words
    
    if not expected_words:
        return {"score": 0.5, "justification": "No meaningful expected words to compare"}
    
    overlap = len(response_words & expected_words) / len(expected_words)
    
    if overlap >= 0.7:
        return {"score": 1, "justification": f"High keyword overlap ({overlap:.2f})"}
    elif overlap >= 0.4:
        return {"score": 0.7, "justification": f"Good keyword overlap ({overlap:.2f})"}
    elif overlap >= 0.2:
        return {"score": 0.4, "justification": f"Partial keyword overlap ({overlap:.2f})"}
    else:
        return {"score": 0, "justification": f"Low keyword overlap ({overlap:.2f})"}

# Create code-based scorers
length_scorer = CodeBasedScorer(name="response_length", func=response_length_scorer)
contact_scorer = CodeBasedScorer(name="includes_contact_info", func=contact_info_scorer)
keyword_scorer = CodeBasedScorer(name="policy_keywords", func=policy_keyword_scorer)
```

For more details, see: [Custom Scorers](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-scorers)

### How to Evaluate Different Components

Different parts of your GenAI application require different evaluation approaches:

#### Retrieval Components
Focus on information retrieval quality:

```python
retrieval_scorers = [
    RetrievalGroundedness(),  # Are responses grounded in retrieved docs?
    RelevanceToQuery(),       # Do retrieved docs match the query?
    Guidelines(
        name="retrieval_coverage",
        guidelines="Retrieved information covers all aspects needed to answer the question"
    )
]
```

#### Generation Components  
Evaluate response quality and accuracy:

```python
generation_scorers = [
    AnswerCorrectness(),      # Factual accuracy
    Faithfulness(),           # Consistency with source material
    Coherence(),              # Logical flow and readability
    Guidelines(
        name="tone_appropriateness", 
        guidelines="Professional and helpful tone appropriate for customer service"
    )
]
```

#### End-to-End System
Assess overall user experience:

```python
system_scorers = [
    Safety(),                 # Content safety
    professional_tone_scorer, # Professional communication
    completeness_scorer,      # Complete responses
    length_scorer,            # Appropriate length
    contact_scorer            # Includes contact info when needed
]
```

#### Business Logic Components
Evaluate compliance and consistency:

```python
business_scorers = [
    brand_compliance_scorer,  # Brand guideline adherence
    policy_accuracy_scorer,   # Policy information accuracy
    Guidelines(
        name="consistency_check",
        guidelines="Response is consistent with company policies and previous interactions"
    )
]
```

#### Comprehensive Scorer Portfolio

Combine different types for complete evaluation:

```python
# Complete scorer suite for Unity Airways
unity_airways_scorers = [
    # Predefined LLM scorers
    RelevanceToQuery(),
    Safety(),
    AnswerCorrectness(),
    
    # Custom LLM scorers
    professional_tone_scorer,
    brand_compliance_scorer,
    empathy_scorer,
    
    # Code-based scorers
    length_scorer,
    contact_scorer,
    keyword_scorer
]
```


---

## 4. Human Feedback {#human-feedback}

Human feedback is essential for evaluating GenAI applications as it captures the user experience and subjective qualities that automated metrics may miss. MLflow 3+ provides comprehensive tools for collecting, analyzing, and integrating human feedback into your evaluation workflow.

### Understanding the Feedback Data Model

The MLflow feedback data model provides a structured approach to capturing and analyzing human judgments:

#### Core Feedback Structure

```python
# Basic feedback record structure
feedback_record = {
    "trace_id": "unique_trace_identifier",
    "feedback_type": "thumbs_up_down",  # or "rating", "categorical", "text"
    "feedback_value": 1,  # 1 for positive, -1 for negative, or rating scale
    "feedback_text": "Response was helpful and accurate",
    "user_id": "user_123",
    "timestamp": "2024-01-15T10:30:00Z",
    "metadata": {
        "source": "web_ui",
        "session_id": "session_456",
        "user_context": "first_time_user"
    }
}
```

#### Feedback Types

1. **Binary Feedback**: Simple thumbs up/down or yes/no responses
2. **Rating Scales**: 1-5 stars, 1-10 numerical ratings
3. **Categorical**: Predefined categories like "helpful", "accurate", "polite"
4. **Text Feedback**: Open-ended comments and suggestions
5. **Comparative**: Preference between multiple responses

#### Feedback Metadata

Rich metadata helps analyze feedback patterns:

```python
feedback_metadata = {
    "user_demographics": {
        "experience_level": "frequent_flyer",
        "age_group": "35-44",
        "preferred_language": "en"
    },
    "interaction_context": {
        "channel": "web_chat",
        "session_duration": 180,
        "previous_interactions": 3,
        "issue_complexity": "medium"
    },
    "response_characteristics": {
        "response_time_ms": 2500,
        "response_length": 85,
        "included_links": true
    }
}
```

### End User Feedback Collection

#### Using the Databricks Review App

The Databricks Review App provides a streamlined interface for collecting structured feedback from domain experts and reviewers:

```python
import mlflow.genai.labeling as labeling

# Create a labeling session for customer satisfaction review
customer_satisfaction_session = labeling.create_labeling_session(
    name="Unity Airways Customer Satisfaction Review",
    description="Collect feedback on customer service response quality",
    assigned_users=[
        "customer_service_manager@unityairways.com",
        "quality_assurance@unityairways.com"
    ]
)

# Add traces to review
recent_traces = mlflow.search_traces(
    filter_string="attributes.status = 'OK'",
    order_by=["attributes.timestamp_ms DESC"],
    max_results=100
)

customer_satisfaction_session.add_traces(recent_traces)

# Create specialized review session for policy accuracy
policy_review_session = labeling.create_labeling_session(
    name="Policy Accuracy Review",
    description="Expert review of policy information accuracy",
    assigned_users=["policy_expert@unityairways.com"]
)

# Filter traces that mention policies
policy_traces = mlflow.search_traces(
    filter_string="attributes.status = 'OK' AND tags.category = 'policy'"
)

policy_review_session.add_traces(policy_traces)
```

#### Embedding Feedback in Applications

Collect feedback directly in your customer-facing applications:

```python
import mlflow
import time

# Log feedback alongside traces
@mlflow.trace
def handle_customer_feedback(trace_id, feedback_type, feedback_value, feedback_text=None, user_context=None):
    """Collect and log customer feedback for a specific interaction."""
    
    feedback_data = {
        "feedback_type": feedback_type,
        "feedback_value": feedback_value,
        "feedback_text": feedback_text,
        "timestamp": time.time(),
        "user_context": user_context or {}
    }
    
    # Log feedback as trace metadata
    mlflow.log_metadata(feedback_data)
    
    # Also log to dedicated feedback tracking
    mlflow.log_metric(f"feedback_{feedback_type}", feedback_value)
    
    return {"status": "feedback_recorded", "trace_id": trace_id}

# Example usage in customer service chat interface
def process_chat_feedback(interaction_id, rating, comment, user_profile):
    """Process feedback from chat interface."""
    
    feedback_result = handle_customer_feedback(
        trace_id=interaction_id,
        feedback_type="rating_1_5",
        feedback_value=rating,
        feedback_text=comment,
        user_context={
            "customer_tier": user_profile.get("tier", "standard"),
            "interaction_channel": "web_chat",
            "issue_resolved": rating >= 4
        }
    )
    
    return feedback_result

# Example: Customer clicks 4-star rating with comment
process_chat_feedback(
    interaction_id="trace_12345",
    rating=4,
    comment="Helpful response but took a while to get specific information",
    user_profile={"tier": "gold", "frequent_flyer": True}
)
```

#### Programmatic Feedback Collection

Collect feedback through APIs or batch processing:

```python
def collect_survey_feedback(survey_responses):
    """Process feedback from customer satisfaction surveys."""
    
    feedback_records = []
    
    for response in survey_responses:
        # Map survey responses to feedback format
        feedback_record = {
            "trace_id": response["interaction_id"],
            "feedback_type": "survey_response",
            "feedback_value": response["overall_satisfaction"],
            "feedback_text": response.get("comments", ""),
            "user_id": response["customer_id"],
            "timestamp": response["survey_completed_at"],
            "metadata": {
                "survey_id": response["survey_id"],
                "response_time": response["time_to_complete"],
                "question_responses": response["detailed_ratings"]
            }
        }
        
        feedback_records.append(feedback_record)
    
    return feedback_records

def integrate_crm_feedback(crm_data):
    """Integrate feedback from CRM system."""
    
    feedback_records = []
    
    for ticket in crm_data:
        if ticket["interaction_type"] == "chat" and ticket["satisfaction_score"]:
            feedback_record = {
                "trace_id": ticket["chat_trace_id"],
                "feedback_type": "crm_satisfaction",
                "feedback_value": ticket["satisfaction_score"],
                "feedback_text": ticket["customer_comments"],
                "user_id": ticket["customer_id"],
                "metadata": {
                    "ticket_id": ticket["id"],
                    "resolution_time": ticket["resolution_minutes"],
                    "escalated": ticket["was_escalated"]
                }
            }
            
            feedback_records.append(feedback_record)
    
    return feedback_records
```

### Analyzing Feedback Data

#### Feedback Analytics and Pattern Recognition

```python
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

def analyze_feedback_patterns(feedback_data):
    """Comprehensive analysis of user feedback patterns."""
    
    # Convert to DataFrame for analysis
    df = pd.DataFrame(feedback_data)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['date'] = df['timestamp'].dt.date
    
    analysis_results = {}
    
    # 1. Overall feedback distribution
    feedback_distribution = df['feedback_value'].value_counts().sort_index()
    analysis_results['distribution'] = feedback_distribution
    
    # 2. Feedback trends over time
    daily_avg_feedback = df.groupby('date')['feedback_value'].mean()
    analysis_results['trends'] = daily_avg_feedback
    
    # 3. Feedback by user segments
    if 'user_context' in df.columns:
        df['customer_tier'] = df['user_context'].apply(lambda x: x.get('customer_tier', 'standard'))
        tier_feedback = df.groupby('customer_tier')['feedback_value'].mean()
        analysis_results['by_tier'] = tier_feedback
    
    # 4. Response characteristics correlation
    if 'response_length' in df.columns:
        length_correlation = df['response_length'].corr(df['feedback_value'])
        analysis_results['length_correlation'] = length_correlation
    
    # 5. Text feedback sentiment analysis
    negative_feedback = df[df['feedback_value'] < 3]['feedback_text'].dropna()
    positive_feedback = df[df['feedback_value'] >= 4]['feedback_text'].dropna()
    
    analysis_results['negative_themes'] = extract_themes(negative_feedback)
    analysis_results['positive_themes'] = extract_themes(positive_feedback)
    
    return analysis_results

def extract_themes(feedback_texts):
    """Extract common themes from feedback text."""
    if len(feedback_texts) == 0:
        return []
    
    # Simple keyword extraction (in practice, use more sophisticated NLP)
    all_text = ' '.join(feedback_texts.astype(str)).lower()
    
    # Common complaint/praise keywords
    keywords = {
        'speed': ['fast', 'quick', 'slow', 'delay', 'wait'],
        'accuracy': ['correct', 'wrong', 'accurate', 'mistake', 'error'],
        'helpfulness': ['helpful', 'useless', 'useful', 'unhelpful'],
        'politeness': ['polite', 'rude', 'friendly', 'professional']
    }
    
    themes = {}
    for theme, words in keywords.items():
        count = sum(1 for word in words if word in all_text)
        if count > 0:
            themes[theme] = count
    
    return sorted(themes.items(), key=lambda x: x[1], reverse=True)

def visualize_feedback_analysis(analysis_results):
    """Create interactive visualizations of feedback analysis."""
    
    # Create subplots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            'Feedback Distribution',
            'Feedback Trends Over Time',
            'Feedback by Customer Tier',
            'Common Themes'
        ],
        specs=[[{"type": "bar"}, {"type": "scatter"}],
               [{"type": "bar"}, {"type": "bar"}]]
    )
    
    # Feedback distribution
    dist_data = analysis_results['distribution']
    fig.add_trace(
        go.Bar(x=dist_data.index, y=dist_data.values, name='Distribution'),
        row=1, col=1
    )
    
    # Trends over time
    trends_data = analysis_results['trends']
    fig.add_trace(
        go.Scatter(x=trends_data.index, y=trends_data.values, mode='lines+markers', name='Trends'),
        row=1, col=2
    )
    
    # Feedback by tier
    if 'by_tier' in analysis_results:
        tier_data = analysis_results['by_tier']
        fig.add_trace(
            go.Bar(x=tier_data.index, y=tier_data.values, name='By Tier'),
            row=2, col=1
        )
    
    # Common themes
    if analysis_results['negative_themes']:
        themes, counts = zip(*analysis_results['negative_themes'][:5])
        fig.add_trace(
            go.Bar(x=list(themes), y=list(counts), name='Negative Themes'),
            row=2, col=2
        )
    
    fig.update_layout(height=800, showlegend=False, title_text="Feedback Analysis Dashboard")
    return fig
```

#### Correlating Feedback with Automated Scores

```python
def correlate_feedback_with_automated_scores(traces_with_feedback, automated_scores):
    """Find correlations between automated scores and human feedback."""
    
    # Merge feedback with automated scores
    merged_data = pd.merge(traces_with_feedback, automated_scores, on='trace_id')
    
    # Calculate correlations
    score_columns = [col for col in merged_data.columns if 'score' in col.lower() and col != 'feedback_value']
    
    correlations = {}
    for score_col in score_columns:
        correlation = merged_data[score_col].corr(merged_data['feedback_value'])
        correlations[score_col] = correlation
    
    # Identify best predictors of human satisfaction
    best_predictors = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)
    
    # Find discrepancies (high automated score, low human feedback)
    discrepancies = merged_data[
        (merged_data['automated_score_avg'] >= 0.8) & 
        (merged_data['feedback_value'] <= 2)
    ]
    
    return {
        'correlations': correlations,
        'best_predictors': best_predictors,
        'discrepancies': discrepancies,
        'alignment_score': sum(abs(corr) for corr in correlations.values()) / len(correlations)
    }

def identify_feedback_insights(correlation_analysis):
    """Generate actionable insights from feedback correlation analysis."""
    
    insights = []
    
    # Strong positive correlations
    strong_positive = [(name, corr) for name, corr in correlation_analysis['best_predictors'] if corr > 0.7]
    if strong_positive:
        insights.append(f"Strong predictors of satisfaction: {', '.join([name for name, _ in strong_positive])}")
    
    # Weak correlations (potential issues)
    weak_correlations = [(name, corr) for name, corr in correlation_analysis['correlations'].items() if abs(corr) < 0.3]
    if weak_correlations:
        insights.append(f"Automated scores not aligned with user feedback: {', '.join([name for name, _ in weak_correlations])}")
    
    # Discrepancy analysis
    if len(correlation_analysis['discrepancies']) > 0:
        insights.append(f"Found {len(correlation_analysis['discrepancies'])} cases where automated scores were high but user feedback was low")
    
    return insights
```


---

## 5. Evaluation Runs {#evaluation-runs}

Evaluation runs are the execution engine of GenAI evaluation in MLflow. They systematically apply scorers to evaluation datasets, track results, and enable comparison across model versions and time periods.

### Understanding Evaluation Runs

An evaluation run in MLflow 3+ consists of:

1. **Input Dataset**: The evaluation dataset containing test examples with inputs and expected outputs
2. **Model/Function**: The GenAI application being evaluated (wrapped as a callable function)
3. **Scorers**: The set of evaluation metrics to apply (LLM-based and code-based)
4. **Execution Context**: Environment, parameters, and metadata for reproducibility
5. **Results**: Scores, traces, artifacts, and detailed analysis generated

#### Evaluation Run Lifecycle

1. **Setup**: Define dataset, model function, and scorers
2. **Execution**: MLflow runs the model on each dataset example and applies scorers
3. **Collection**: Results are aggregated and stored with detailed traces
4. **Analysis**: Results can be viewed in MLflow UI or analyzed programmatically
5. **Comparison**: Multiple runs can be compared for A/B testing and improvement tracking

### Creating Evaluation Runs

#### Basic Evaluation Run

```python
import mlflow.genai

# Define your model function (must be callable)
@mlflow.trace
def unity_airways_customer_service(question: str) -> dict:
    """Unity Airways customer service agent function."""
    # Your GenAI application logic here
    response = generate_customer_service_response(question)
    return {"response": response}

# Define comprehensive scorers
scorers = [
    # Predefined LLM scorers
    RelevanceToQuery(),
    Safety(),
    AnswerCorrectness(),
    
    # Custom scorers
    professional_tone_scorer,
    brand_compliance_scorer,
    
    # Code-based scorers
    length_scorer,
    contact_scorer
]

# Run evaluation
eval_results = mlflow.genai.evaluate(
    data=evaluation_dataset,  # Your evaluation dataset
    predict_fn=unity_airways_customer_service,
    scorers=scorers,
    run_name="Unity Airways Customer Service v1.0"
)

print(f"Evaluation completed. Run ID: {eval_results.run_id}")
```

#### Advanced Evaluation Configuration

```python
# More sophisticated evaluation with custom configuration
eval_results = mlflow.genai.evaluate(
    data=evaluation_dataset,
    predict_fn=unity_airways_customer_service,
    scorers=scorers,
    run_name="Unity Airways v2.0 - Enhanced Safety",
    
    # Evaluation parameters
    evaluator_config={
        "timeout": 120,  # Timeout per example in seconds
        "max_workers": 10,  # Parallel evaluation workers
        "retry_attempts": 3,  # Retry failed evaluations
        "batch_size": 5  # Process examples in batches
    },
    
    # Additional metadata for tracking
    extra_metrics={
        "model_version": "2.0",
        "dataset_version": "1.3",
        "evaluation_date": "2024-01-15",
        "evaluator": "qa_team"
    },
    
    # Custom experiment tracking
    experiment_name="Unity Airways Customer Service Evaluation"
)
```

#### Evaluation with Custom Data Processing

```python
def preprocess_evaluation_data(dataset):
    """Custom preprocessing for evaluation data."""
    processed_examples = []
    
    for example in dataset:
        # Add context or modify inputs as needed
        processed_example = {
            "inputs": {
                "question": example["inputs"]["question"],
                "context": "Unity Airways Customer Service",
                "user_tier": example.get("user_tier", "standard")
            },
            "expected": example["expected"]
        }
        processed_examples.append(processed_example)
    
    return processed_examples

# Use preprocessed data
processed_dataset = preprocess_evaluation_data(evaluation_dataset)

eval_results = mlflow.genai.evaluate(
    data=processed_dataset,
    predict_fn=unity_airways_customer_service,
    scorers=scorers,
    run_name="Unity Airways v2.0 - Preprocessed Data"
)
```

### View and Interpret Results

#### MLflow UI Analysis

```python
# Generate URL to view results in MLflow UI
def get_evaluation_url(run_id, experiment_name="Default"):
    """Generate URL to view evaluation results in MLflow UI."""
    tracking_uri = mlflow.get_tracking_uri()
    try:
        experiment = mlflow.get_experiment_by_name(experiment_name)
        experiment_id = experiment.experiment_id
    except:
        experiment_id = "0"  # Default experiment
    
    return f"{tracking_uri}/#/experiments/{experiment_id}/runs/{run_id}"

# View results
print(f"View results in MLflow UI: {get_evaluation_url(eval_results.run_id, 'Unity Airways Customer Service Evaluation')}")
```

#### Programmatic Results Analysis

```python
def analyze_evaluation_results(eval_results):
    """Comprehensive analysis of evaluation results."""
    
    # Get run details
    run = mlflow.get_run(eval_results.run_id)
    metrics = run.data.metrics
    params = run.data.params
    
    # Extract and organize scores
    score_summary = {}
    for metric_name, value in metrics.items():
        if '/mean' in metric_name:
            scorer_name = metric_name.replace('metrics/', '').replace('/mean', '')
            score_summary[scorer_name] = value
    
    # Calculate performance statistics
    scores_list = list(score_summary.values())
    performance_stats = {
        'overall_score': sum(scores_list) / len(scores_list) if scores_list else 0,
        'min_score': min(scores_list) if scores_list else 0,
        'max_score': max(scores_list) if scores_list else 0,
        'score_variance': sum((s - sum(scores_list)/len(scores_list))**2 for s in scores_list) / len(scores_list) if scores_list else 0
    }
    
    # Identify top and bottom performers
    sorted_scores = sorted(score_summary.items(), key=lambda x: x[1])
    worst_scorer = sorted_scores[0] if sorted_scores else None
    best_scorer = sorted_scores[-1] if sorted_scores else None
    
    # Get detailed traces for further analysis
    evaluation_traces = mlflow.search_traces(run_id=eval_results.run_id)
    
    return {
        'run_id': eval_results.run_id,
        'run_name': run.info.run_name,
        'performance_stats': performance_stats,
        'individual_scores': score_summary,
        'best_performer': best_scorer,
        'worst_performer': worst_scorer,
        'total_examples': len(evaluation_traces),
        'run_duration': run.info.end_time - run.info.start_time if run.info.end_time else None
    }

# Analyze results
analysis = analyze_evaluation_results(eval_results)
print(f"Overall Score: {analysis['performance_stats']['overall_score']:.3f}")
print(f"Best Scorer: {analysis['best_performer']}")
print(f"Worst Scorer: {analysis['worst_performer']}")
print(f"Total Examples Evaluated: {analysis['total_examples']}")
```

#### Detailed Failure Analysis

```python
def analyze_evaluation_failures(eval_results, failure_threshold=0.5):
    """Identify and analyze failed evaluations for improvement insights."""
    
    # Get evaluation traces
    evaluation_traces = mlflow.search_traces(run_id=eval_results.run_id)
    
    failures = []
    successes = []
    
    for trace in evaluation_traces:
        if hasattr(trace, 'assessments') and trace.assessments:
            trace_failures = []
            trace_successes = []
            
            for assessment in trace.assessments:
                score_value = assessment.score if hasattr(assessment, 'score') else None
                feedback_value = assessment.feedback.value if hasattr(assessment, 'feedback') else None
                
                # Determine if this assessment failed
                failed = False
                if score_value is not None and score_value < failure_threshold:
                    failed = True
                elif feedback_value == 'no':
                    failed = True
                
                assessment_data = {
                    'trace_id': trace.trace_id,
                    'input': trace.request,
                    'output': trace.response,
                    'scorer_name': assessment.name,
                    'score': score_value,
                    'feedback': feedback_value,
                    'rationale': assessment.rationale if hasattr(assessment, 'rationale') else None
                }
                
                if failed:
                    trace_failures.append(assessment_data)
                else:
                    trace_successes.append(assessment_data)
            
            if trace_failures:
                failures.extend(trace_failures)
            if trace_successes:
                successes.extend(trace_successes)
    
    # Group failures by scorer
    failures_by_scorer = {}
    for failure in failures:
        scorer = failure['scorer_name']
        if scorer not in failures_by_scorer:
            failures_by_scorer[scorer] = []
        failures_by_scorer[scorer].append(failure)
    
    # Analyze failure patterns
    failure_analysis = {
        'total_failures': len(failures),
        'total_successes': len(successes),
        'failure_rate': len(failures) / (len(failures) + len(successes)) if (len(failures) + len(successes)) > 0 else 0,
        'failures_by_scorer': failures_by_scorer,
        'most_problematic_scorer': max(failures_by_scorer.items(), key=lambda x: len(x[1]))[0] if failures_by_scorer else None
    }
    
    return failure_analysis

def generate_improvement_recommendations(failure_analysis):
    """Generate actionable recommendations based on failure analysis."""
    
    recommendations = []
    
    # High failure rate recommendations
    if failure_analysis['failure_rate'] > 0.3:
        recommendations.append("High failure rate detected. Consider reviewing model training data and prompts.")
    
    # Scorer-specific recommendations
    for scorer, failures in failure_analysis['failures_by_scorer'].items():
        failure_count = len(failures)
        if failure_count > 5:
            # Analyze common failure patterns
            rationales = [f['rationale'] for f in failures if f['rationale']]
            common_issues = analyze_common_failure_themes(rationales)
            
            recommendations.append(f"{scorer}: {failure_count} failures detected. Common issues: {', '.join(common_issues[:3])}")
    
    return recommendations

def analyze_common_failure_themes(rationales):
    """Extract common themes from failure rationales."""
    if not rationales:
        return []
    
    # Simple keyword analysis (in practice, use more sophisticated NLP)
    all_text = ' '.join(rationales).lower()
    
    common_themes = {
        'length': ['too short', 'too long', 'length'],
        'relevance': ['not relevant', 'off-topic', 'doesn\'t answer'],
        'accuracy': ['incorrect', 'wrong', 'inaccurate'],
        'tone': ['unprofessional', 'inappropriate tone', 'rude'],
        'completeness': ['incomplete', 'missing', 'partial']
    }
    
    found_themes = []
    for theme, keywords in common_themes.items():
        if any(keyword in all_text for keyword in keywords):
            found_themes.append(theme)
    
    return found_themes

# Analyze failures and get recommendations
failure_analysis = analyze_evaluation_failures(eval_results)
recommendations = generate_improvement_recommendations(failure_analysis)

print("Failure Analysis:")
print(f"- Total failures: {failure_analysis['total_failures']}")
print(f"- Failure rate: {failure_analysis['failure_rate']:.2%}")
print(f"- Most problematic scorer: {failure_analysis['most_problematic_scorer']}")

print("\nRecommendations:")
for rec in recommendations:
    print(f"- {rec}")
```

#### Comparing Evaluation Runs

```python
def compare_evaluation_runs(run_id_1, run_id_2, run_name_1="Version 1", run_name_2="Version 2"):
    """Compare two evaluation runs for A/B testing and improvement tracking."""
    
    # Get run data
    run_1 = mlflow.get_run(run_id_1)
    run_2 = mlflow.get_run(run_id_2)
    
    # Extract metrics (focusing on mean scores)
    metrics_1 = {k.replace('metrics/', '').replace('/mean', ''): v 
                 for k, v in run_1.data.metrics.items() if '/mean' in k}
    metrics_2 = {k.replace('metrics/', '').replace('/mean', ''): v 
                 for k, v in run_2.data.metrics.items() if '/mean' in k}
    
    # Calculate improvements
    comparison_data = []
    for metric in set(metrics_1.keys()) & set(metrics_2.keys()):
        v1_score = metrics_1[metric]
        v2_score = metrics_2[metric]
        improvement = v2_score - v1_score
        improvement_pct = (improvement / v1_score * 100) if v1_score != 0 else 0
        
        comparison_data.append({
            'metric': metric,
            f'{run_name_1}_score': v1_score,
            f'{run_name_2}_score': v2_score,
            'improvement': improvement,
            'improvement_pct': improvement_pct,
            'significant': abs(improvement_pct) >= 5  # 5% threshold for significance
        })
    
    # Overall comparison
    avg_improvement = sum(item['improvement'] for item in comparison_data) / len(comparison_data) if comparison_data else 0
    significant_improvements = sum(1 for item in comparison_data if item['improvement'] > 0 and item['significant'])
    significant_regressions = sum(1 for item in comparison_data if item['improvement'] < 0 and item['significant'])
    
    comparison_summary = {
        'comparison_data': pd.DataFrame(comparison_data),
        'avg_improvement': avg_improvement,
        'significant_improvements': significant_improvements,
        'significant_regressions': significant_regressions,
        'net_improvement': significant_improvements - significant_regressions
    }
    
    return comparison_summary

# Compare two evaluation runs
comparison = compare_evaluation_runs(
    eval_results_v1.run_id, 
    eval_results_v2.run_id, 
    "Version 1.0", 
    "Version 2.0"
)

print("Evaluation Run Comparison:")
print(f"Average improvement: {comparison['avg_improvement']:+.3f}")
print(f"Significant improvements: {comparison['significant_improvements']}")
print(f"Significant regressions: {comparison['significant_regressions']}")
print(f"Net improvement: {comparison['net_improvement']}")

# Display detailed comparison
display(comparison['comparison_data'])
```


---

## 6. Best Practices {#best-practices}

This section consolidates key best practices for evaluating GenAI applications with MLflow 3+ on Databricks, drawn from both theoretical foundations and production experience.

### Dataset Management Best Practices

#### 1. Dataset Diversity and Quality

```python
# Ensure comprehensive coverage across dimensions
dataset_coverage_framework = {
    'functional_coverage': {
        'booking_scenarios': ['new_booking', 'modification', 'cancellation'],
        'policy_questions': ['baggage', 'refund', 'change_fees', 'special_assistance'],
        'support_types': ['information', 'troubleshooting', 'complaints']
    },
    'user_coverage': {
        'customer_types': ['first_time', 'frequent_flyer', 'business', 'leisure'],
        'experience_levels': ['novice', 'intermediate', 'expert'],
        'emotional_states': ['calm', 'frustrated', 'urgent', 'confused']
    },
    'complexity_coverage': {
        'simple': 'Single, straightforward questions',
        'medium': 'Multi-part questions or edge cases',
        'complex': 'Ambiguous or multi-step scenarios'
    }
}

def validate_dataset_coverage(dataset, coverage_framework):
    """Ensure balanced representation across key dimensions."""
    coverage_report = {}
    
    for dimension, categories in coverage_framework.items():
        if isinstance(categories, dict):
            dimension_coverage = {}
            for category, subcategories in categories.items():
                if isinstance(subcategories, list):
                    for subcat in subcategories:
                        count = sum(1 for example in dataset 
                                  if subcat in str(example).lower())
                        dimension_coverage[subcat] = count
                else:
                    count = sum(1 for example in dataset 
                              if category in str(example).lower())
                    dimension_coverage[category] = count
            coverage_report[dimension] = dimension_coverage
    
    return coverage_report
```

#### 2. Version Control and Lineage

```python
# Implement systematic dataset versioning
def create_versioned_dataset(base_dataset, version_info, changes_description):
    """Create a new dataset version with proper lineage tracking."""
    
    dataset_metadata = {
        'version': version_info['version'],
        'parent_version': version_info.get('parent_version'),
        'creation_date': datetime.now().isoformat(),
        'changes': changes_description,
        'size': len(base_dataset),
        'quality_metrics': calculate_dataset_quality_metrics(base_dataset)
    }
    
    versioned_dataset = mlflow.genai.datasets.create_dataset(
        uc_table_name=f"catalog.schema.unity_airways_eval_v{version_info['version']}",
        name=f"Unity Airways Evaluation Dataset v{version_info['version']}",
        description=f"Version {version_info['version']}: {changes_description}",
        metadata=dataset_metadata
    )
    
    versioned_dataset.merge_records(base_dataset)
    return versioned_dataset

# Track dataset lineage
def track_dataset_lineage(dataset_versions):
    """Maintain clear lineage between dataset versions."""
    lineage_graph = {}
    
    for version_info in dataset_versions:
        version = version_info['version']
        parent = version_info.get('parent_version')
        
        lineage_graph[version] = {
            'parent': parent,
            'changes': version_info['changes'],
            'quality_delta': version_info.get('quality_delta', 0)
        }
    
    return lineage_graph
```

#### 3. Ground Truth Management

```python
# Establish robust ground truth validation
def validate_ground_truth_quality(examples, validation_criteria):
    """Multi-reviewer validation for ground truth examples."""
    
    validation_results = []
    
    for example in examples:
        validation_score = 0
        validation_notes = []
        
        # Check completeness
        if all(key in example for key in ['inputs', 'expected']):
            validation_score += 0.25
        else:
            validation_notes.append("Missing required fields")
        
        # Check clarity
        if len(example['inputs'].get('question', '')) > 10:
            validation_score += 0.25
        else:
            validation_notes.append("Question too short or unclear")
        
        # Check expected response quality
        expected_response = example['expected'].get('expected_response', '')
        if len(expected_response) > 20 and len(expected_response.split()) > 5:
            validation_score += 0.25
        else:
            validation_notes.append("Expected response insufficient")
        
        # Check consistency
        if 'category' in example['expected']:
            validation_score += 0.25
        else:
            validation_notes.append("Missing categorization")
        
        validation_results.append({
            'example_id': example.get('id', 'unknown'),
            'validation_score': validation_score,
            'needs_review': validation_score < 0.8,
            'notes': validation_notes
        })
    
    return validation_results
```

### Scorer Selection and Configuration

#### 1. Balanced Scorer Portfolio

```python
# Recommended scorer mix for comprehensive evaluation
def create_comprehensive_scorer_suite(domain_requirements):
    """Create a balanced portfolio of scorers for comprehensive evaluation."""
    
    scorer_portfolio = {
        # Foundation scorers (always include)
        'foundation': [
            RelevanceToQuery(),  # Basic relevance
            Safety(),           # Content safety
            AnswerCorrectness() # Factual accuracy
        ],
        
        # Domain-specific scorers
        'domain_specific': [],
        
        # Quality scorers
        'quality': [
            Coherence(),        # Logical flow
            Faithfulness()      # Source consistency
        ],
        
        # Business logic scorers
        'business': [],
        
        # Performance scorers
        'performance': []
    }
    
    # Add domain-specific scorers based on requirements
    if domain_requirements.get('customer_service'):
        scorer_portfolio['domain_specific'].extend([
            Guidelines(name="professional_tone", 
                      guidelines="Use professional, courteous language"),
            Guidelines(name="empathy", 
                      guidelines="Show understanding and compassion for customer concerns")
        ])
    
    if domain_requirements.get('brand_compliance'):
        scorer_portfolio['business'].append(
            Guidelines(name="brand_guidelines",
                      guidelines="Follow company brand and communication guidelines")
        )
    
    if domain_requirements.get('performance_monitoring'):
        scorer_portfolio['performance'].extend([
            CodeBasedScorer(name="response_length", func=response_length_checker),
            CodeBasedScorer(name="response_time", func=response_time_checker)
        ])
    
    # Flatten the portfolio
    all_scorers = []
    for category, scorers in scorer_portfolio.items():
        all_scorers.extend(scorers)
    
    return all_scorers, scorer_portfolio

# Example usage
domain_requirements = {
    'customer_service': True,
    'brand_compliance': True,
    'performance_monitoring': True
}

comprehensive_scorers, scorer_breakdown = create_comprehensive_scorer_suite(domain_requirements)
```

#### 2. Scorer Calibration and Validation

```python
# Implement scorer calibration process
def calibrate_llm_scorers(scorers, calibration_dataset, human_ratings):
    """Calibrate LLM scorers against human judgment."""
    
    calibration_results = {}
    
    for scorer in scorers:
        if hasattr(scorer, 'guidelines') or 'Guidelines' in str(type(scorer)):
            # Run scorer on calibration dataset
            scorer_results = []
            for example in calibration_dataset:
                # Simulate scorer execution
                result = scorer.score(example['input'], example['output'])
                scorer_results.append(result['score'])
            
            # Compare with human ratings
            correlation = calculate_correlation(scorer_results, human_ratings)
            agreement_rate = calculate_agreement_rate(scorer_results, human_ratings)
            
            calibration_results[scorer.name] = {
                'correlation': correlation,
                'agreement_rate': agreement_rate,
                'needs_adjustment': correlation < 0.7 or agreement_rate < 0.8
            }
    
    return calibration_results

def adjust_scorer_guidelines(scorer, calibration_feedback):
    """Adjust scorer guidelines based on calibration results."""
    
    if calibration_feedback['needs_adjustment']:
        # Analyze disagreement patterns
        disagreement_patterns = analyze_disagreement_patterns(calibration_feedback)
        
        # Generate guideline improvements
        improved_guidelines = enhance_guidelines(
            scorer.guidelines, 
            disagreement_patterns
        )
        
        # Create updated scorer
        updated_scorer = Guidelines(
            name=f"{scorer.name}_v2",
            guidelines=improved_guidelines
        )
        
        return updated_scorer
    
    return scorer
```

### Human Feedback Integration

#### 1. Multi-Channel Feedback Strategy

```python
# Implement comprehensive feedback collection
feedback_collection_strategy = {
    'real_time_feedback': {
        'channels': ['web_ui', 'mobile_app', 'chat_interface'],
        'types': ['thumbs_up_down', 'star_rating', 'quick_feedback'],
        'frequency': 'every_interaction'
    },
    'periodic_surveys': {
        'channels': ['email', 'in_app_survey'],
        'types': ['detailed_satisfaction', 'feature_feedback'],
        'frequency': 'weekly'
    },
    'expert_review': {
        'channels': ['databricks_review_app', 'manual_review'],
        'types': ['quality_assessment', 'policy_compliance'],
        'frequency': 'continuous'
    },
    'a_b_testing': {
        'channels': ['controlled_experiments'],
        'types': ['preference_comparison', 'effectiveness_metrics'],
        'frequency': 'per_release'
    }
}

def implement_feedback_collection(strategy):
    """Implement multi-channel feedback collection system."""
    
    feedback_collectors = {}
    
    for method, config in strategy.items():
        collector = create_feedback_collector(
            method=method,
            channels=config['channels'],
            types=config['types'],
            frequency=config['frequency']
        )
        feedback_collectors[method] = collector
    
    return feedback_collectors
```

#### 2. Feedback Quality Assurance

```python
# Implement feedback quality controls
def validate_feedback_quality(feedback_data):
    """Validate and filter feedback for quality and reliability."""
    
    quality_filters = {
        'completeness': lambda f: all(key in f for key in ['trace_id', 'feedback_value', 'timestamp']),
        'recency': lambda f: (time.time() - f['timestamp']) < 30 * 24 * 3600,  # 30 days
        'consistency': lambda f: validate_user_consistency(f),
        'authenticity': lambda f: not detect_spam_feedback(f)
    }
    
    filtered_feedback = []
    quality_report = {'total': len(feedback_data), 'filtered': 0, 'reasons': {}}
    
    for feedback in feedback_data:
        passed_filters = True
        failed_reasons = []
        
        for filter_name, filter_func in quality_filters.items():
            if not filter_func(feedback):
                passed_filters = False
                failed_reasons.append(filter_name)
        
        if passed_filters:
            filtered_feedback.append(feedback)
        else:
            quality_report['filtered'] += 1
            for reason in failed_reasons:
                quality_report['reasons'][reason] = quality_report['reasons'].get(reason, 0) + 1
    
    return filtered_feedback, quality_report
```

### Evaluation Run Management

#### 1. Systematic Evaluation Cadence

```python
# Establish evaluation schedules
evaluation_schedules = {
    'continuous_monitoring': {
        'frequency': 'hourly',
        'scope': 'production_sample',
        'scorers': ['safety', 'basic_quality'],
        'alert_thresholds': {'safety': 0.95, 'quality': 0.8}
    },
    'daily_health_check': {
        'frequency': 'daily',
        'scope': 'representative_sample',
        'scorers': 'comprehensive_lite',
        'alert_thresholds': {'overall': 0.75}
    },
    'weekly_deep_dive': {
        'frequency': 'weekly',
        'scope': 'full_evaluation_dataset',
        'scorers': 'comprehensive_full',
        'analysis': 'trend_analysis'
    },
    'release_validation': {
        'frequency': 'pre_release',
        'scope': 'full_evaluation_dataset',
        'scorers': 'comprehensive_full',
        'requirements': 'regression_testing'
    }
}

def schedule_evaluation_runs(schedules):
    """Implement automated evaluation scheduling."""
    
    scheduled_runs = {}
    
    for schedule_name, config in schedules.items():
        scheduler = create_evaluation_scheduler(
            name=schedule_name,
            frequency=config['frequency'],
            scope=config['scope'],
            scorers=config['scorers'],
            thresholds=config.get('alert_thresholds', {}),
            analysis_type=config.get('analysis', 'basic')
        )
        scheduled_runs[schedule_name] = scheduler
    
    return scheduled_runs
```

#### 2. Regression Testing Framework

```python
# Implement systematic regression testing
def create_regression_testing_framework():
    """Create comprehensive regression testing for GenAI evaluation."""
    
    regression_framework = {
        'baseline_management': {
            'golden_dataset': 'curated_high_quality_examples',
            'baseline_scores': 'previous_release_scores',
            'acceptance_criteria': 'minimum_performance_thresholds'
        },
        'regression_detection': {
            'statistical_tests': ['t_test', 'wilcoxon_signed_rank'],
            'practical_significance': 'effect_size_thresholds',
            'domain_specific_rules': 'business_logic_constraints'
        },
        'automated_alerts': {
            'immediate_alerts': 'critical_regressions',
            'daily_reports': 'trend_analysis',
            'release_gates': 'go_no_go_decisions'
        }
    }
    
    return regression_framework

def execute_regression_testing(new_run_id, baseline_run_id, framework):
    """Execute comprehensive regression testing."""
    
    # Compare runs
    comparison = compare_evaluation_runs(new_run_id, baseline_run_id)
    
    # Apply regression detection rules
    regressions = detect_regressions(comparison, framework['regression_detection'])
    
    # Generate alerts if needed
    if regressions['critical_regressions']:
        send_immediate_alerts(regressions['critical_regressions'])
    
    # Create regression report
    regression_report = create_regression_report(regressions, comparison)
    
    return regression_report
```

### Production Deployment Best Practices

#### 1. Staged Evaluation Deployment

```python
# Multi-stage deployment strategy
deployment_pipeline = {
    'development': {
        'evaluation_scope': 'full_comprehensive',
        'dataset': 'development_dataset',
        'frequency': 'every_commit',
        'gates': 'basic_quality_checks'
    },
    'staging': {
        'evaluation_scope': 'production_like',
        'dataset': 'staging_dataset',
        'frequency': 'every_deployment',
        'gates': 'regression_testing'
    },
    'canary': {
        'evaluation_scope': 'limited_production',
        'dataset': 'production_sample',
        'frequency': 'continuous',
        'gates': 'real_time_monitoring'
    },
    'production': {
        'evaluation_scope': 'full_production',
        'dataset': 'production_traffic',
        'frequency': 'continuous',
        'gates': 'comprehensive_monitoring'
    }
}

def implement_staged_deployment(pipeline):
    """Implement staged evaluation deployment."""
    
    deployment_stages = {}
    
    for stage, config in pipeline.items():
        stage_implementation = create_deployment_stage(
            stage_name=stage,
            evaluation_scope=config['evaluation_scope'],
            dataset=config['dataset'],
            frequency=config['frequency'],
            quality_gates=config['gates']
        )
        deployment_stages[stage] = stage_implementation
    
    return deployment_stages
```

#### 2. Continuous Learning and Improvement

```python
# Implement continuous improvement loop
def create_continuous_improvement_system():
    """Create system for continuous learning and improvement."""
    
    improvement_system = {
        'data_collection': {
            'production_traces': 'continuous_collection',
            'user_feedback': 'real_time_integration',
            'performance_metrics': 'automated_monitoring'
        },
        'analysis_and_insights': {
            'pattern_detection': 'automated_analysis',
            'trend_identification': 'statistical_monitoring',
            'root_cause_analysis': 'failure_investigation'
        },
        'improvement_actions': {
            'dataset_updates': 'automated_curation',
            'scorer_adjustments': 'calibration_updates',
            'model_recommendations': 'improvement_suggestions'
        },
        'validation_and_deployment': {
            'improvement_testing': 'a_b_testing',
            'gradual_rollout': 'canary_deployment',
            'impact_measurement': 'effectiveness_tracking'
        }
    }
    
    return improvement_system

def execute_improvement_cycle(improvement_system, cycle_frequency='weekly'):
    """Execute continuous improvement cycle."""
    
    # Collect and analyze recent data
    recent_data = collect_recent_evaluation_data(cycle_frequency)
    insights = analyze_evaluation_insights(recent_data)
    
    # Generate improvement recommendations
    recommendations = generate_improvement_recommendations(insights)
    
    # Implement approved improvements
    implemented_changes = implement_improvements(recommendations)
    
    # Validate improvements
    validation_results = validate_improvements(implemented_changes)
    
    return {
        'insights': insights,
        'recommendations': recommendations,
        'implemented_changes': implemented_changes,
        'validation_results': validation_results
    }
```

### Summary

This comprehensive notebook demonstrates best practices for evaluating GenAI applications using MLflow 3+ on Databricks. Key takeaways include:

1. **Holistic Evaluation Strategy**: Combine automated scoring with human feedback for comprehensive assessment
2. **Systematic Dataset Management**: Use diverse, version-controlled evaluation datasets with proper lineage tracking
3. **Balanced Scorer Portfolio**: Mix predefined LLM scorers, custom guidelines-based judges, and code-based scorers
4. **Comprehensive Human Feedback**: Implement multi-channel feedback collection with quality assurance
5. **Structured Evaluation Runs**: Use systematic evaluation schedules with proper analysis and comparison
6. **Production Integration**: Deploy evaluation in stages with continuous monitoring and improvement
7. **Continuous Learning**: Implement feedback loops for ongoing enhancement and adaptation

By following these practices, organizations can build robust, reliable GenAI applications that consistently deliver value to users while maintaining safety and quality standards.

### Next Steps

1. **Implement Evaluation Pipeline**: Set up automated evaluation runs for your GenAI application
2. **Establish Feedback Collection**: Integrate human feedback collection into your application workflow  
3. **Create Custom Scorers**: Develop domain-specific evaluation metrics tailored to your use case
4. **Monitor Production Performance**: Implement real-time quality monitoring and alerting
5. **Build Improvement Processes**: Create systematic workflows for addressing evaluation insights and implementing improvements

### Additional Resources

- [MLflow GenAI Evaluation Documentation](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/)
- [Predefined Judge Scorers](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/predefined-judge-scorers)
- [Custom Judge Guidelines](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-judge/meets-guidelines)
- [Custom Scorers](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-scorers)
- [Databricks GenAI Best Practices](https://docs.databricks.com/aws/en/mlflow3/genai/)


In [0]:
%pip install --upgrade "mlflow[databricks]>=3.1" plotly
dbutils.library.restartPython()


# Chapter 6: Evaluating GenAI Applications within MLflow 3+ on Databricks

**Learning Objective:** Understand and apply evaluation techniques for GenAI applications using MLflow.

## Table of Contents
1. [The Modern Evaluation Landscape in MLflow](#modern-evaluation)
2. [Creating & Managing Evaluation Datasets](#evaluation-datasets)
3. [Scores](#scores)
4. [Human Feedback](#human-feedback)
5. [Evaluation Runs](#evaluation-runs)
6. [Best Practices](#best-practices)

## Use Case: Unity Airways Customer Service Agent

This notebook demonstrates best practices for evaluating a GenAI-powered customer service agent for Unity Airways (a fictional airline). The agent helps customers with:
- Flight bookings and modifications
- Policy questions (baggage, refunds, cancellations)
- General customer support inquiries

We'll evaluate using both structured booking data and unstructured FAQ/QA datasets, demonstrating the full spectrum of GenAI evaluation techniques in MLflow 3+.


---

## 1. The Modern Evaluation Landscape in MLflow {#modern-evaluation}

GenAI applications represent a paradigm shift from traditional ML models. Unlike classic models that produce structured, deterministic outputs, GenAI systems generate natural language responses that must be evaluated for qualities like relevance, helpfulness, factual accuracy, and user satisfaction.

### Why Traditional Metrics Miss GenAI Application Behaviour

Traditional ML evaluation metrics (accuracy, precision, recall, F1-score) were designed for classification and regression tasks with clear ground truth labels. These metrics fall short for GenAI applications because:

1. **Deterministic vs. Generative Outputs**: Traditional models produce fixed outputs for given inputs, while GenAI models generate variable, creative responses
2. **Structured vs. Unstructured Data**: Traditional metrics work with numerical or categorical outputs, not natural language text
3. **Single Correct Answer vs. Multiple Valid Responses**: GenAI tasks often have many acceptable answers, making binary accuracy insufficient
4. **Context and Nuance**: Traditional metrics don't capture semantic meaning, tone, helpfulness, or user experience
5. **Safety and Ethics**: GenAI outputs must be evaluated for harmful content, bias, and policy compliance

### What are the Components to Evaluate

GenAI applications typically consist of multiple components that require different evaluation approaches:

1. **Retrieval Components**: 
   - Retrieval accuracy and relevance
   - Document ranking quality
   - Coverage of relevant information

2. **Generation Components**:
   - Factual correctness and groundedness
   - Relevance to user query
   - Coherence and fluency
   - Tone and style appropriateness

3. **End-to-End System**:
   - User experience and satisfaction
   - Task completion effectiveness
   - Safety and policy compliance
   - Latency and performance

4. **Business Logic**:
   - Adherence to guidelines and policies
   - Consistency across similar queries
   - Integration with downstream systems

### Evaluation Modes: Direct Evaluation vs Answer Sheet Evaluation

**Direct Evaluation:**
- Assesses model outputs directly against criteria or guidelines
- Uses LLM judges to evaluate qualities like helpfulness, relevance, safety
- Suitable for open-ended tasks without single correct answers
- Examples: Chatbot responses, creative writing, summarization

**Answer Sheet Evaluation:**
- Compares model outputs to curated reference answers
- Uses exact match, semantic similarity, or custom comparison functions
- Suitable for tasks with clear correct answers
- Examples: Question answering, fact extraction, classification


---

## 2. Creating & Managing Evaluation Datasets {#evaluation-datasets}

Evaluation datasets are the foundation of GenAI application testing. MLflow 3+ provides powerful tools for creating, managing, and versioning evaluation datasets that enable systematic testing and continuous improvement.

### Dataset Types and Sources

**1. Production Trace Datasets**
- Built from real user interactions captured by MLflow Tracing
- Provides authentic user scenarios and edge cases
- Enables testing against actual production patterns

**2. Curated Test Datasets**
- Manually created examples targeting specific features or edge cases
- Ground truth answers for answer sheet evaluation
- Domain expert validated responses

**3. Synthetic Datasets**
- Generated using LLMs to expand test coverage
- Useful for testing rare scenarios or adversarial cases
- Can simulate different user personas and interaction styles

### Approaches to Building Evaluation Datasets

MLflow 3+ offers several flexible approaches to construct evaluation datasets:

#### Approach 1: Build from Existing Traces

One of the most effective ways to build relevant evaluation datasets is by curating examples from your application's historical interactions captured by MLflow Tracing.

```python
import mlflow
import time

# Search for traces from the last hour
one_hour_ago = int((time.time() - 60 * 60) * 1000)

traces = mlflow.search_traces(
    filter_string=f"attributes.timestamp_ms > {one_hour_ago} AND "
                 f"attributes.status = 'OK'",
    order_by=["attributes.timestamp_ms DESC"],
    max_results=100
)

# Add traces to evaluation dataset
eval_dataset.merge_records(traces)
```

#### Approach 2: Build from Scratch or Import Existing

You can import existing datasets or create examples from scratch. Data must match the evaluation dataset schema:

```python
evaluation_examples = [
    {
        "inputs": {"question": "What is the baggage allowance for international flights?"},
        "expected": {
            "expected_response": "For international flights, you can bring one carry-on bag (22x14x9 inches) and one personal item. Checked baggage allowance varies by fare type.",
            "expected_categories": ["baggage", "international", "policy"]
        }
    },
    {
        "inputs": {"question": "How do I cancel my flight?"},
        "expected": {
            "expected_response": "You can cancel your flight online through Manage My Booking, by calling customer service, or at the airport. Cancellation fees may apply.",
            "expected_categories": ["cancellation", "booking", "policy"]
        }
    }
]

eval_dataset.merge_records(evaluation_examples)
```

#### Approach 3: Synthesize Evaluation Sets

Generate synthetic data to expand testing coverage and create diverse scenarios:

```python
from mlflow.genai.datasets import synthesize_dataset

# Generate synthetic customer service scenarios
synthetic_dataset = synthesize_dataset(
    base_examples=evaluation_examples,
    num_examples=50,
    persona_variations=["frustrated customer", "first-time flyer", "business traveler"],
    scenario_types=["booking", "cancellation", "policy inquiry", "complaint"]
)

eval_dataset.merge_records(synthetic_dataset)
```

#### Approach 4: Domain Expert Labels

Leverage feedback from domain experts captured in MLflow Labeling Sessions:

```python
import mlflow.genai.labeling as labeling

# Get labeling sessions
labeling_sessions = labeling.get_labeling_sessions()

# Sync labeled data to evaluation dataset
for session in labeling_sessions:
    if session.name == "Unity Airways Customer Service Review":
        session.sync(dataset_name="catalog.schema.unity_airways_eval")
```

### Creating MLflow Evaluation Datasets

```python
import mlflow.genai.datasets

# Create evaluation dataset
eval_dataset = mlflow.genai.datasets.create_dataset(
    uc_table_name="catalog.schema.unity_airways_evaluation_dataset",
    name="Unity Airways Customer Service Evaluation",
    description="Comprehensive evaluation dataset for Unity Airways customer service agent"
)
```

For more details, see: [Build Evaluation Dataset Documentation](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/build-eval-dataset)


---

## 3. Scores {#scores}

Scoring is the heart of GenAI evaluation. MLflow 3+ provides a comprehensive framework for assessing GenAI applications using both automated and human-informed evaluation methods.

### How Scores Work

Scores in MLflow GenAI evaluation are functions that take model inputs and outputs and return a quantitative or qualitative assessment. They can be:

1. **Numerical scores** (0-1, 1-5, etc.) for quantitative metrics
2. **Boolean scores** (pass/fail) for binary criteria
3. **Categorical scores** (good/fair/poor) for qualitative assessments
4. **Structured feedback** with scores and rationales

### Types of Scorers

#### 1. LLM-Based Scorers

Use large language models as judges to evaluate outputs for complex qualities that require understanding of context, semantics, and nuance.

**Advantages:**
- Can evaluate subjective qualities (helpfulness, tone, coherence)
- Understand context and nuance
- Provide detailed rationales
- Scale to evaluate large datasets

**Disadvantages:**
- Can be inconsistent
- Require careful prompt engineering
- May have biases
- Higher latency and cost

#### 2. Code-Based Scorers

Use deterministic Python functions to evaluate outputs based on specific rules, patterns, or calculations.

**Advantages:**
- Deterministic and consistent
- Fast execution
- No additional LLM costs
- Easy to debug and modify

**Disadvantages:**
- Limited to rule-based evaluation
- Cannot assess subjective qualities
- May miss nuanced cases
- Require manual rule definition

### Using LLM-Based Scorers

MLflow 3+ provides both predefined and custom LLM-based scorers:

#### Predefined LLM Scorers

```python
from mlflow.genai.scorers import (
    RetrievalGroundedness,    # Checks if response is grounded in retrieved context
    RelevanceToQuery,         # Evaluates relevance to user query  
    Safety,                   # Detects harmful or inappropriate content
    AnswerCorrectness,        # Compares against ground truth answers
    Faithfulness,             # Checks factual consistency
    AnswerRelevance           # Evaluates answer relevance
)

# Use predefined scorers
scorers = [
    RetrievalGroundedness(),
    RelevanceToQuery(), 
    Safety(),
    AnswerCorrectness()
]
```

For complete list and details, see: [Predefined Judge Scorers](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/predefined-judge-scorers)

#### Custom LLM Scorers

**Guidelines-Based Judges:**
Evaluate outputs against specific guidelines or policies.

```python
from mlflow.genai.scorers import Guidelines

# Custom guidelines-based scorer
professional_tone_scorer = Guidelines(
    name="professional_tone",
    guidelines="The response must use professional, courteous language appropriate for customer service. Avoid casual expressions, slang, or overly informal tone."
)

brand_compliance_scorer = Guidelines(
    name="brand_compliance", 
    guidelines="The response must follow Unity Airways brand guidelines: mention the airline name when relevant, use positive language, and maintain helpful tone."
)
```

**Prompt-Based Judges:**
Create custom evaluation logic using prompt engineering.

```python
from mlflow.genai.scorers import PromptBasedJudge

# Custom prompt-based scorer
completeness_scorer = PromptBasedJudge(
    name="response_completeness",
    prompt="""
    Evaluate if the customer service response completely addresses the customer's question.
    
    Customer Question: {question}
    Agent Response: {response}
    
    Consider:
    1. Does the response directly answer the question?
    2. Are all parts of multi-part questions addressed?
    3. Is sufficient detail provided?
    4. Are next steps clearly explained?
    
    Rate on a scale of 1-5 where:
    1 = Completely inadequate, major parts of question ignored
    2 = Partially addresses question but missing important elements  
    3 = Addresses main question but lacks some detail
    4 = Addresses question well with good detail
    5 = Completely and thoroughly addresses all aspects
    
    Provide your rating and brief explanation.
    """
)
```

For more details, see: 
- [Custom Judge Guidelines](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-judge/meets-guidelines)
- [Create Prompt Judge](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-judge/create-prompt-judge)

### Using Code-Based Scorers

Code-based scorers provide deterministic evaluation using Python functions:

```python
from mlflow.genai.scorers import CodeBasedScorer

def response_length_checker(prediction, **kwargs):
    """Score based on response length appropriateness."""
    response = prediction if isinstance(prediction, str) else prediction.get('response', '')
    word_count = len(response.split())
    
    if word_count < 10:
        return {"score": 0, "justification": f"Response too short ({word_count} words)"}
    elif word_count > 200:
        return {"score": 0, "justification": f"Response too long ({word_count} words)"}
    else:
        return {"score": 1, "justification": f"Appropriate length ({word_count} words)"}

def contact_info_scorer(prediction, **kwargs):
    """Check if response includes contact information when appropriate."""
    response = prediction if isinstance(prediction, str) else prediction.get('response', '')
    contact_keywords = ["call", "phone", "email", "contact"]
    if any(keyword in response.lower() for keyword in contact_keywords):
        if "1-800" in response or "@unityairways.com" in response:
            return {"score": 1, "justification": "Includes proper contact info"}
        else:
            return {"score": 0, "justification": "Mentions contact but no specific info"}
    return {"score": 0.5, "justification": "No contact info mentioned"}

# Create code-based scorers
length_scorer = CodeBasedScorer(name="response_length", func=response_length_checker)
contact_scorer = CodeBasedScorer(name="includes_contact_info", func=contact_info_scorer)
```

For more details, see: [Custom Scorers](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-scorers)

### How to Evaluate Different Components

Different parts of your GenAI application require different evaluation approaches:

#### Retrieval Components
```python
retrieval_scorers = [
    RetrievalGroundedness(),  # Are responses grounded in retrieved docs?
    RelevanceToQuery(),       # Do retrieved docs match the query?
]
```

#### Generation Components  
```python
generation_scorers = [
    AnswerCorrectness(),      # Factual accuracy
    Faithfulness(),           # Consistency with source material
    Guidelines(name="tone", guidelines="Professional and helpful tone"),
]
```

#### End-to-End System
```python
system_scorers = [
    Safety(),                 # Content safety
    Guidelines(name="completeness", guidelines="Complete and actionable response"),
    CodeBasedScorer(name="latency", func=latency_checker),
]
```


---

## 4. Human Feedback {#human-feedback}

Human feedback is essential for evaluating GenAI applications as it captures the user experience and subjective qualities that automated metrics may miss. MLflow 3+ provides comprehensive tools for collecting, analyzing, and integrating human feedback into your evaluation workflow.

### Understanding the Feedback Data Model

The MLflow feedback data model supports structured collection and analysis of human judgments:

```python
# Feedback record structure
feedback_record = {
    "trace_id": "unique_trace_identifier",
    "feedback_type": "thumbs_up_down",  # or "rating", "categorical", "text"
    "feedback_value": 1,  # 1 for positive, -1 for negative, or rating scale
    "feedback_text": "Response was helpful and accurate",
    "user_id": "user_123",
    "timestamp": "2024-01-15T10:30:00Z",
    "metadata": {
        "source": "web_ui",
        "session_id": "session_456"
    }
}
```

### Types of Human Feedback

#### 1. End User Feedback
Direct feedback from application users during normal usage:
- **Thumbs up/down**: Simple binary feedback
- **Star ratings**: 1-5 star quality ratings  
- **Free text**: Open-ended comments and suggestions
- **Categorical**: Predefined categories (helpful, accurate, polite, etc.)

#### 2. Expert Feedback
Structured feedback from domain experts or reviewers:
- **Quality assessments**: Detailed evaluation against criteria
- **Ground truth validation**: Confirming correct answers
- **Policy compliance**: Checking adherence to guidelines
- **Comparative evaluation**: Ranking multiple responses

#### 3. A/B Testing Feedback
Comparative feedback between different model versions:
- **Preference ratings**: Which response is better?
- **Specific criteria comparison**: Better accuracy, helpfulness, etc.
- **Conversion metrics**: Task completion rates

### End User Feedback Collection

#### Using the Databricks Review App

The Databricks Review App provides a streamlined interface for collecting structured feedback:

```python
import mlflow.genai.labeling as labeling

# Create a labeling session
session = labeling.create_labeling_session(
    name="Unity Airways Customer Satisfaction Review",
    description="Collect feedback on customer service responses",
    assigned_users=["reviewer1@company.com", "reviewer2@company.com"]
)

# Add traces to review
traces_to_review = mlflow.search_traces(
    filter_string="attributes.status = 'OK'",
    max_results=50
)

session.add_traces(traces_to_review)
```

#### Embedding Feedback in Applications

Collect feedback directly in your application interface:

```python
import mlflow

# Log feedback alongside traces
@mlflow.trace
def handle_user_feedback(trace_id, feedback_type, feedback_value, feedback_text=None):
    """Collect and log user feedback for a specific interaction."""
    
    feedback_data = {
        "feedback_type": feedback_type,
        "feedback_value": feedback_value,
        "feedback_text": feedback_text,
        "timestamp": time.time()
    }
    
    # Log feedback as trace metadata
    mlflow.log_metadata(feedback_data)
    
    return {"status": "feedback_recorded"}

# Example usage in a web app
# When user clicks thumbs up:
handle_user_feedback(
    trace_id="trace_123", 
    feedback_type="thumbs_up_down", 
    feedback_value=1,
    feedback_text="Very helpful response!"
)
```

### Analyzing Feedback Data

#### Feedback Analytics

```python
def analyze_feedback_patterns(feedback_data):
    """Analyze patterns in user feedback."""
    
    # Feedback distribution
    feedback_distribution = feedback_data['feedback_value'].value_counts()
    
    # Feedback by time period
    feedback_data['date'] = pd.to_datetime(feedback_data['timestamp']).dt.date
    feedback_trends = feedback_data.groupby('date')['feedback_value'].mean()
    
    # Common themes in text feedback
    negative_feedback = feedback_data[feedback_data['feedback_value'] == -1]['feedback_text']
    positive_feedback = feedback_data[feedback_data['feedback_value'] == 1]['feedback_text']
    
    return {
        'distribution': feedback_distribution,
        'trends': feedback_trends,
        'negative_themes': negative_feedback.tolist(),
        'positive_themes': positive_feedback.tolist()
    }
```

#### Correlating Feedback with Model Performance

```python
def correlate_feedback_with_scores(traces_with_feedback):
    """Find correlations between automated scores and human feedback."""
    
    # Extract automated scores
    score_columns = [col for col in traces_with_feedback.columns if 'score' in col.lower()]
    
    correlations = {}
    for score_col in score_columns:
        correlation = traces_with_feedback[score_col].corr(traces_with_feedback['feedback_value'])
        correlations[score_col] = correlation
    
    # Identify which automated scores best predict human satisfaction
    best_predictors = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)
    
    return correlations, best_predictors
```

### Feedback-Driven Improvements

#### Using Feedback for Model Training

```python
def create_training_data_from_feedback(traces_with_feedback):
    """Convert feedback into training examples."""
    
    # Create preference pairs for RLHF
    positive_examples = traces_with_feedback[traces_with_feedback['feedback_value'] == 1]
    negative_examples = traces_with_feedback[traces_with_feedback['feedback_value'] == -1]
    
    training_data = []
    
    for _, pos_example in positive_examples.iterrows():
        for _, neg_example in negative_examples.iterrows():
            # Create preference pair
            training_data.append({
                'input': pos_example['input'],
                'chosen': pos_example['output'],
                'rejected': neg_example['output'],
                'preference_strength': abs(pos_example['feedback_value'] - neg_example['feedback_value'])
            })
    
    return training_data
```

#### Feedback-Based Dataset Curation

```python
def curate_dataset_from_feedback(traces_with_feedback, min_rating=4):
    """Create high-quality dataset from well-rated examples."""
    
    # Filter for high-quality examples
    high_quality = traces_with_feedback[traces_with_feedback['feedback_value'] >= min_rating]
    
    # Create evaluation dataset
    curated_examples = []
    for _, row in high_quality.iterrows():
        curated_examples.append({
            'inputs': row['input'],
            'expected': {
                'expected_response': row['output'],
                'quality_score': row['feedback_value']
            }
        })
    
    return curated_examples
```


---

## 5. Evaluation Runs {#evaluation-runs}

Evaluation runs are the execution engine of GenAI evaluation in MLflow. They systematically apply scorers to evaluation datasets, track results, and enable comparison across model versions and time periods.

### Understanding Evaluation Runs

An evaluation run in MLflow 3+ consists of:

1. **Input Dataset**: The evaluation dataset containing test examples
2. **Model/Function**: The GenAI application being evaluated  
3. **Scorers**: The set of evaluation metrics to apply
4. **Execution Context**: Environment, parameters, and metadata
5. **Results**: Scores, traces, and artifacts generated

### Creating Evaluation Runs

#### Basic Evaluation Run

```python
import mlflow.genai

# Define your model function
@mlflow.trace
def customer_service_agent(question: str) -> dict:
    # Your GenAI application logic
    response = generate_response(question)
    return {"response": response}

# Define scorers
scorers = [
    RelevanceToQuery(),
    Safety(),
    Guidelines(name="professional_tone", guidelines="Use professional language")
]

# Run evaluation
eval_results = mlflow.genai.evaluate(
    data=evaluation_dataset,
    predict_fn=customer_service_agent,
    scorers=scorers,
    run_name="Unity Airways Customer Service v1.0"
)
```

#### Advanced Evaluation Configuration

```python
# More sophisticated evaluation with custom configuration
eval_results = mlflow.genai.evaluate(
    data=evaluation_dataset,
    predict_fn=customer_service_agent,
    scorers=scorers,
    run_name="Unity Airways v2.0 - Safety Enhanced",
    
    # Evaluation parameters
    evaluator_config={
        "timeout": 120,  # Timeout per example in seconds
        "max_workers": 5,  # Parallel evaluation workers
        "retry_attempts": 3  # Retry failed evaluations
    },
    
    # Additional metadata
    extra_metrics={
        "model_version": "2.0",
        "dataset_version": "1.2",
        "evaluation_date": "2024-01-15"
    }
)
```

### View and Interpret Results

#### MLflow UI Analysis

```python
# Generate URL to view results in MLflow UI
def get_evaluation_url(run_id):
    tracking_uri = mlflow.get_tracking_uri()
    return f"{tracking_uri}/#/experiments/{mlflow.get_experiment_by_name('Unity Airways').experiment_id}/runs/{run_id}"

print(f"View results: {get_evaluation_url(eval_results.run_id)}")
```

#### Programmatic Results Analysis

```python
def analyze_evaluation_results(eval_results):
    """Comprehensive analysis of evaluation results."""
    
    # Get run metrics
    run = mlflow.get_run(eval_results.run_id)
    metrics = run.data.metrics
    
    # Aggregate score analysis
    score_summary = {}
    for metric_name, value in metrics.items():
        if '/mean' in metric_name:
            scorer_name = metric_name.replace('metrics/', '').replace('/mean', '')
            score_summary[scorer_name] = value
    
    # Identify top and bottom performers
    sorted_scores = sorted(score_summary.items(), key=lambda x: x[1])
    worst_scorer = sorted_scores[0] if sorted_scores else None
    best_scorer = sorted_scores[-1] if sorted_scores else None
    
    # Calculate overall performance
    overall_score = sum(score_summary.values()) / len(score_summary) if score_summary else 0
    
    return {
        'overall_score': overall_score,
        'individual_scores': score_summary,
        'best_performer': best_scorer,
        'worst_performer': worst_scorer,
        'total_examples': len(eval_results.traces) if hasattr(eval_results, 'traces') else 0
    }

# Analyze results
analysis = analyze_evaluation_results(eval_results)
print(f"Overall Score: {analysis['overall_score']:.3f}")
print(f"Best Scorer: {analysis['best_performer']}")
print(f"Worst Scorer: {analysis['worst_performer']}")
```

#### Failure Analysis

```python
def analyze_failures(evaluation_traces):
    """Identify and analyze failed evaluations."""
    
    failures = []
    
    for trace in evaluation_traces:
        # Check for evaluation failures
        if hasattr(trace, 'assessments'):
            for assessment in trace.assessments:
                if assessment.feedback.value == 'no' or assessment.score < 0.5:
                    failures.append({
                        'trace_id': trace.trace_id,
                        'input': trace.request,
                        'output': trace.response,
                        'failed_scorer': assessment.name,
                        'score': assessment.score,
                        'rationale': assessment.rationale
                    })
    
    # Group failures by scorer
    failures_by_scorer = {}
    for failure in failures:
        scorer = failure['failed_scorer']
        if scorer not in failures_by_scorer:
            failures_by_scorer[scorer] = []
        failures_by_scorer[scorer].append(failure)
    
    return failures_by_scorer

# Analyze failures
failures = analyze_failures(evaluation_traces)
for scorer, failed_examples in failures.items():
    print(f"\n{scorer}: {len(failed_examples)} failures")
    if failed_examples:
        print(f"Example failure: {failed_examples[0]['rationale'][:100]}...")
```

### Comparing Evaluation Runs

#### Version Comparison

```python
def compare_evaluation_runs(run_id_1, run_id_2, run_name_1="V1", run_name_2="V2"):
    """Compare two evaluation runs."""
    
    # Get run data
    run_1 = mlflow.get_run(run_id_1)
    run_2 = mlflow.get_run(run_id_2)
    
    # Extract metrics
    metrics_1 = {k.replace('metrics/', ''): v for k, v in run_1.data.metrics.items() if '/mean' in k}
    metrics_2 = {k.replace('metrics/', ''): v for k, v in run_2.data.metrics.items() if '/mean' in k}
    
    # Calculate improvements
    comparison = []
    for metric in set(metrics_1.keys()) & set(metrics_2.keys()):
        improvement = metrics_2[metric] - metrics_1[metric]
        comparison.append({
            'metric': metric.replace('/mean', ''),
            f'{run_name_1}_score': metrics_1[metric],
            f'{run_name_2}_score': metrics_2[metric],
            'improvement': improvement,
            'improvement_pct': (improvement / metrics_1[metric] * 100) if metrics_1[metric] != 0 else 0
        })
    
    return pd.DataFrame(comparison)

# Compare runs
comparison_df = compare_evaluation_runs(eval_results_v1.run_id, eval_results_v2.run_id)
display(comparison_df)
```

#### Trend Analysis

```python
def analyze_evaluation_trends(experiment_name, days=30):
    """Analyze evaluation trends over time."""
    
    # Get experiment
    experiment = mlflow.get_experiment_by_name(experiment_name)
    
    # Search runs from last N days
    cutoff_time = int((time.time() - days * 24 * 60 * 60) * 1000)
    
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=f"attributes.start_time > {cutoff_time}",
        order_by=["attributes.start_time DESC"]
    )
    
    # Extract trend data
    trend_data = []
    for _, run in runs.iterrows():
        metrics = {k: v for k, v in run.items() if k.startswith('metrics.') and '/mean' in k}
        trend_data.append({
            'run_id': run['run_id'],
            'run_name': run['tags.mlflow.runName'],
            'start_time': pd.to_datetime(run['start_time']),
            **metrics
        })
    
    return pd.DataFrame(trend_data)

# Analyze trends
trends_df = analyze_evaluation_trends("Unity Airways Evaluation")
display(trends_df.head())
```
